# Facial Emotion Recognition — Training Notebook
Use this notebook to experiment with training runs, visualize curves, and debug on small subsets before running the full `train.py` script.

In [5]:
import sys
sys.path.append('..')  # so we can import from src/
import os
os.chdir("..")   # move up from notebooks/ to project root
print(os.getcwd())  # confirm it's now Face/
import matplotlib.pyplot as plt
from src.data.dataset import FERdata
from src.model.model import FERModel

C:\Users\Hp\Desktop\Face


## 1. Load data

In [7]:
data = FERdata()
train_data, val_data, test_data = data.load()

print("Class indices:", train_data.class_indices)
print("Train samples:", train_data.samples)
print("Val samples:", val_data.samples)
print("Test samples:", test_data.samples)

Found 22968 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.
Found 5741 images belonging to 7 classes.
Class indices: {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}
Train samples: 22968
Val samples: 5741
Test samples: 7178


## 2. Build and compile model

In [8]:
fer_model = FERModel()
fer_model.load(activation="relu")
model = fer_model.compile()
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 46, 46, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 23, 23, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 21, 21, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 10, 10, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 8, 8, 128)           │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 4, 4, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 2048)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │         131,136 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 7)                   │             231 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 226,119 (883.28 KB)

 Trainable params: 226,119 (883.28 KB)

 Non-trainable params: 0 (0.00 B)

## 3. Quick debug run (small epochs, sanity check only)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=3
)

Epoch 1/3
718/718 ━━━━━━━━━━━━━━━━━━━━ 21s 27ms/step - accuracy: 0.2877 - loss: 1.7534 - val_accuracy: 0.3830 - val_loss: 1.6065
Epoch 2/3
718/718 ━━━━━━━━━━━━━━━━━━━━ 21s 29ms/step - accuracy: 0.4268 - loss: 1.4846 - val_accuracy: 0.4541 - val_loss: 1.4206
Epoch 3/3
 25/718 ━━━━━━━━━━━━━━━━━━━━ 17s 25ms/step - accuracy: 0.5012 - loss: 1.3433

## 4. Visualize training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='train loss')
axes[0].plot(history.history['val_loss'], label='val loss')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='train acc')
axes[1].plot(history.history['val_accuracy'], label='val acc')
axes[1].set_title('Accuracy')
axes[1].legend()

plt.show()

## 5. Evaluate on test set

In [ ]:
test_loss, test_acc = model.evaluate(test_data)
print(f"Test accuracy: {test_acc:.4f}")

---
**Note:** Once you're satisfied with the architecture/hyperparameters here, move the finalized training loop into `src/training/train.py` for the real, reproducible training run (with checkpointing and early stopping).